# Alakoro FiberSense — Processadores Avançados C++20

Este notebook demonstra o uso dos processadores avançados implementados em C++20 e expostos via pybind11:

- **Filtros Butterworth** (`butterworth_lowpass`, `butterworth_highpass`, `butterworth_bandpass`)
- **FFT e PSD** (`magnitude_spectrum`, `psd`)
- **CWT** (`cwt`) com wavelets Morlet e Ricker

Também mostramos como esses processadores se integram ao `LFDASProcessor` e ao `SignatureValidator`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.simulation import SignatureGenerator, WellGeometry, AcquisitionConfig
from src.io.dasdae import DASDAEAdapter
from src.io.alakoro_spool import AlakoroPatch
from src.processing.advanced_processors import (
    butterworth_lowpass,
    butterworth_highpass,
    butterworth_bandpass,
    magnitude_spectrum,
    psd,
    cwt,
)
from src.processing.lfdas_processor import LFDASProcessor
from src.validation.signature_validator import SignatureValidator

## 1. Geração de dados sintéticos DAS

Usamos o `SignatureGenerator` para criar uma assinatura de *valve chatter*.

In [ ]:
well = WellGeometry(depth_top=0, depth_bottom=3000, n_channels=3000)
acq = AcquisitionConfig(sampling_rate_hz=1000, trace_interval_s=2.0, duration_s=3600)
generator = SignatureGenerator(well, acq)

sig = generator.generate_valve_chatter(valve_depth=1400.0)
das_data = sig['das']

print('Shape DAS:', das_data.shape)
print('Taxa de amostragem:', acq.sampling_rate_hz, 'Hz')

## 2. Filtros Butterworth

Convertemos o array NumPy para `AlakoroPatch` e aplicamos os filtros C++20.

In [ ]:
patch = AlakoroPatch(
    DASDAEAdapter.array_to_patch(das_data.astype(np.float64), modality='das'),
    modality='das'
)

lowpass = butterworth_lowpass(patch, sample_rate_hz=1000.0, cutoff_hz=50.0)
highpass = butterworth_highpass(patch, sample_rate_hz=1000.0, cutoff_hz=10.0)
bandpass = butterworth_bandpass(patch, sample_rate_hz=1000.0, low_hz=10.0, high_hz=100.0)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].imshow(das_data[:, 1300:1500].T, aspect='auto', vmin=-1, vmax=1)
axes[0, 0].set_title('Original')
axes[0, 1].imshow(lowpass.data[:, 1300:1500].T, aspect='auto', vmin=-1, vmax=1)
axes[0, 1].set_title('Lowpass (<50 Hz)')
axes[1, 0].imshow(highpass.data[:, 1300:1500].T, aspect='auto', vmin=-1, vmax=1)
axes[1, 0].set_title('Highpass (>10 Hz)')
axes[1, 1].imshow(bandpass.data[:, 1300:1500].T, aspect='auto', vmin=-1, vmax=1)
axes[1, 1].set_title('Bandpass (10–100 Hz)')
plt.tight_layout()

## 3. Magnitude Spectrum e PSD

Calculamos a densidade espectral de potência por canal. A saída é um array 1D com `n_freq * n_channels` elementos, onde `n_freq = n_times // 2 + 1`.

In [ ]:
spec = magnitude_spectrum(patch)
spectrum = psd(patch, sample_rate_hz=1000.0)

n_freq = das_data.shape[0] // 2 + 1
freqs = np.linspace(0, 500, n_freq)
psd_per_channel = spectrum.reshape(n_freq, -1)

plt.figure(figsize=(12, 4))
plt.semilogy(freqs, psd_per_channel[:, :3])
plt.xlabel('Frequência (Hz)')
plt.ylabel('PSD')
plt.title('PSD dos primeiros 3 canais')
plt.grid(True)

## 4. CWT — Transformada Wavelet Contínua

A CWT produz uma matriz tempo-frequência (escala) para cada canal. Útil para detectar transientes localizados.

In [ ]:
# Usamos um patch menor para visualização mais rápida
small_data = das_data[:128, 1400:1410].astype(np.float64)
small_patch = AlakoroPatch(
    DASDAEAdapter.array_to_patch(small_data, modality='das'),
    modality='das'
)

scales = [1.0, 2.0, 4.0, 8.0, 16.0]
coefs = cwt(small_patch, scales=scales, sample_rate_hz=1000.0, wavelet='morlet')

print('Número de canais processados:', len(coefs))
print('Shape do primeiro canal:', coefs[0].shape)

plt.figure(figsize=(10, 4))
plt.imshow(coefs[0], aspect='auto', cmap='jet')
plt.colorbar(label='|Coeficiente|')
plt.yticks(range(len(scales)), [f'{s:.0f}' for s in scales])
plt.ylabel('Escala')
plt.xlabel('Amostra de tempo')
plt.title('CWT Morlet — canal 1400')

## 5. LF-DAS com backend C++

O `LFDASProcessor` pode usar o filtro Butterworth C++20 ao invés do scipy. Basta passar `use_cpp_backend=True`.

In [ ]:
lfdas_scipy = LFDASProcessor(cutoff_hz=1.0, refresh_rate_target_s=2.0, use_cpp_backend=False)
lfdas_cpp = LFDASProcessor(cutoff_hz=1.0, refresh_rate_target_s=2.0, use_cpp_backend=True)

result_scipy = lfdas_scipy.process(das_data, trace_interval_s=acq.trace_interval_s)
result_cpp = lfdas_cpp.process(das_data, trace_interval_s=acq.trace_interval_s)

print('Backend scipy:', result_scipy['metadata']['method'])
print('Backend C++:', result_cpp['metadata']['method'])
print('Shape temperatura (C++):', result_cpp['temperature'].shape)
print('Diferença máxima entre backends:', np.max(np.abs(result_scipy['temperature'] - result_cpp['temperature'])))

## 6. SignatureValidator com validações avançadas

Ativando `advanced_checks=True`, o validador utiliza PSD e CWT para verificar conteúdo de frequência e presença de transientes.

In [ ]:
validator = SignatureValidator(well, acq)

# Validação padrão
result_basic = validator.validate_signature(sig)
print(f"Básica: {result_basic['passed']}/{result_basic['total']} passaram")

# Validação avançada (PSD + CWT)
result_advanced = validator.validate_signature(
    sig, advanced_checks=True, sample_rate_hz=1000.0
)
print(f"Avançada: {result_advanced['passed']}/{result_advanced['total']} passaram")

for test in result_advanced['tests']:
    status = '✅' if test['passed'] else '❌'
    print(f'{status} {test["message"]}')

## 7. Pipeline completo: pré-processamento + extração de features para ML

Combina filtros, PSD e CWT para alimentar o `DASFeatureExtractor`.

In [ ]:
from src.ml.features import DASFeatureExtractor

# 1. Pré-processamento
filtered = butterworth_bandpass(patch, sample_rate_hz=1000.0, low_hz=10.0, high_hz=100.0)

# 2. PSD como feature
psd_features = psd(filtered, sample_rate_hz=1000.0)

# 3. Features estatísticas + DAS específicas
features = DASFeatureExtractor().transform(filtered)

print('Shape features extraídas:', features.shape)
print('Primeiros valores:', features[:5])